In [3]:
# Install LangChain if not already present
!pip install langchain langchain-community

In [5]:
from langchain_core.prompts import PromptTemplate

# --- Task 1: Converting Hardcoded f-strings to LangChain Templates ---

# Original hardcoded approach (for reference):
template_string = "Explain {topic} in simple terms for beginners"

# Initialize the PromptTemplate.
# input_variables matches the placeholder in the template_string.
topic_prompt = PromptTemplate(
    input_variables=["topic"],
    template=template_string
)

# Testing the template with a specific input
# This produces the final formatted prompt string
formatted_prompt = topic_prompt.format(topic="Quantum Computing")

print("Generated Prompt:")
print(formatted_prompt)

Generated Prompt:
Explain Quantum Computing in simple terms for beginners


In [6]:
# --- Task 2: Building a Multi-Input Prompt System ---

# Defining a template that accepts three distinct variables
multi_input_template = "Explain {topic} for {audience} in a {tone} tone."

# Initializing the template with multiple input_variables
# Note: No f-strings are used here to maintain strict LangChain standards
multi_prompt = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template=multi_input_template
)

# Test Case 1: AI | beginners | friendly
print("Test Case 1:")
print(multi_prompt.format(topic="AI", audience="beginners", tone="friendly"))

# Test Case 2: Python | kids | fun
print("\nTest Case 2:")
print(multi_prompt.format(topic="Python", audience="kids", tone="fun"))

# Test Case 3: Deep Learning | engineers | technical
print("\nTest Case 3:")
print(multi_prompt.format(topic="Deep Learning", audience="engineers", tone="technical"))

Test Case 1:
Explain AI for beginners in a friendly tone.

Test Case 2:
Explain Python for kids in a fun tone.

Test Case 3:
Explain Deep Learning for engineers in a technical tone.


In [7]:
# --- Task 3: Creating a Prompt Variations Engine ---

# 1. Teaching Variation
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step."
)

# 2. Interview Variation
interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 questions about {topic}."
)

# 3. Storytelling Variation
story_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story."
)

# Running the variations for the topic: Machine Learning
target_topic = "Machine Learning"

print(f"--- Variations for: {target_topic} ---")
print(f"Teaching Mode: {teaching_template.format(topic=target_topic)}")
print(f"Interview Mode: {interview_template.format(topic=target_topic)}")
print(f"Story Mode: {story_template.format(topic=target_topic)}")

--- Variations for: Machine Learning ---
Teaching Mode: Explain Machine Learning clearly step by step.
Interview Mode: Ask 3 questions about Machine Learning.
Story Mode: Explain Machine Learning as a story.


In [9]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# --- Task 4: Building a Chat-based Prompt System with Roles ---

def get_chat_prompt(role_type, topic_input):
    # 1. Define the System Role behavior based on input
    # Roles: teacher, interviewer, motivator
    system_template = "You are a professional {role}. Your goal is to provide expert guidance on the given topic."

    # 2. Create the System and Human message templates
    system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)
    human_template = "Explain the core concepts of {topic}."
    human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

    # 3. Combine into a ChatPromptTemplate
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

    # 4. Format the final messages using the provided inputs
    return chat_prompt.format_prompt(role=role_type, topic=topic_input).to_messages()

# Test Case: Role = Teacher | Topic = Neural Networks
role = 'teacher'
topic = 'Neural Networks'

chat_messages = get_chat_prompt(role, topic)

print(f"--- Chat System for {role.capitalize()} ---")
for message in chat_messages:
    print(f"{type(message).__name__}: {message.content}")

--- Chat System for Teacher ---
SystemMessage: You are a professional teacher. Your goal is to provide expert guidance on the given topic.
HumanMessage: Explain the core concepts of Neural Networks.


In [10]:
# --- Task 5: Input Validation Layer for Audience and Tone ---

def validate_inputs(audience, tone):
    # Defined rules for allowed inputs
    allowed_audiences = ['beginner', 'intermediate', 'expert']
    allowed_tones = ['formal', 'casual', 'fun']

    # Validating Audience
    if audience.lower() not in allowed_audiences:
        print(f"Warning: '{audience}' is invalid. Defaulting to 'beginner'.")
        audience = 'beginner'

    # Validating Tone
    if tone.lower() not in allowed_tones:
        print(f"Warning: '{tone}' is invalid. Defaulting to 'formal'.")
        tone = 'formal'

    return audience, tone

# Testing the validation with an invalid input
test_audience, test_tone = validate_inputs("pro", "cool")
print(f"\nValidated Results -> Audience: {test_audience}, Tone: {test_tone}")


Validated Results -> Audience: beginner, Tone: formal


In [11]:
# --- Task 6: Unified Prompt Generator Application ---

def generate_prompt(topic, audience, tone, style):
    # 1. Internal Validation using our logic from Task 5
    # Ensures the app doesn't crash with non-standard inputs
    audience, tone = validate_inputs(audience, tone)

    # 2. Define the Library of Prompt Templates (Styles)
    # Styles: teaching, interview, storytelling
    style_library = {
        "teaching": "Explain {topic} for {audience} in a {tone} tone. Use a step-by-step teaching style.",
        "interview": "Generate 3 interview questions about {topic} for an {audience} level candidate. Keep the tone {tone}.",
        "storytelling": "Tell a creative story about {topic} for a {audience} audience. The tone should be {tone}."
    }

    # 3. Select the template based on the style provided
    # Defaulting to 'teaching' if the style is not recognized
    selected_template = style_library.get(style.lower(), style_library["teaching"])

    # 4. Initialize the PromptTemplate
    prompt_template = PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template=selected_template
    )

    # 5. Return the final formatted prompt string
    return prompt_template.format(topic=topic, audience=audience, tone=tone)

# Example Output for the Blog/Notebook:
# Topic: Neural Networks | Audience: Beginners | Tone: Fun | Style: Storytelling
final_output = generate_prompt("Neural Networks", "beginner", "fun", "storytelling")

print("--- Generated Prompt App Output ---")
print(final_output)

--- Generated Prompt App Output ---
Tell a creative story about Neural Networks for a beginner audience. The tone should be fun.


In [12]:
# --- Task 7: Template Reusability and Modular Design Test ---

# Define ONE universal template to be reused
universal_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Comprehensive Guide: {topic} | Target: {audience} | Style: {tone}"
)

# List of 5 different input scenarios
test_scenarios = [
    {"topic": "Blockchain", "audience": "investors", "tone": "formal"},
    {"topic": "Photosynthesis", "audience": "5th graders", "tone": "simple"},
    {"topic": "Kubernetes", "audience": "DevOps engineers", "tone": "technical"},
    {"topic": "Yoga", "audience": "seniors", "tone": "gentle"},
    {"topic": "Stock Market", "audience": "teenagers", "tone": "engaging"}
]

print("--- Reusability Test: 5 Different Inputs, 1 Template ---")
for i, scenario in enumerate(test_scenarios, 1):
    formatted = universal_template.format(**scenario)
    print(f"Scenario {i}: {formatted}")

--- Reusability Test: 5 Different Inputs, 1 Template ---
Scenario 1: Comprehensive Guide: Blockchain | Target: investors | Style: formal
Scenario 2: Comprehensive Guide: Photosynthesis | Target: 5th graders | Style: simple
Scenario 3: Comprehensive Guide: Kubernetes | Target: DevOps engineers | Style: technical
Scenario 4: Comprehensive Guide: Yoga | Target: seniors | Style: gentle
Scenario 5: Comprehensive Guide: Stock Market | Target: teenagers | Style: engaging
